<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Background Subtraction</b></h1>
</div>

## Theoretical Foundations

This notebook documents the mathematical model, processing assumptions, controlled ablations, diagnostics, and limitations of the neurointerventional background-subtraction laboratory.
### Technical Context

The image sequence contains a largely static anatomical background and sparse moving interventional tools. The objective is to estimate a stable background representation, isolate temporal residuals associated with guidewire and microcatheter motion, suppress nuisance structure, and produce a binary tool mask.

### Core Background-Subtraction Model

For frame $I_t$ and background estimate $B$,

$$
D_t(x,y)=I_t(x,y)-B(x,y).
$$

The residual $D_t$ is subsequently transformed through intensity normalization, spatial filtering, spectral filtering, morphology, and segmentation.

### Notation and Conventions

| Symbol | Meaning |
|---|---|
| $I_t$ | fluoroscopic frame at time $t$ |
| $B$ | background estimate |
| $D_t$ | background-subtraction residual |
| $G_t$ | manual ground-truth mask |
| $M_t$ | predicted mask |
| $\sigma$ | Gaussian spatial-filter standard deviation |
| $D_0$ | spectral high-pass cutoff |
| $T$ | segmentation threshold |
| $Q$ | field-of-view support mask |

Mask convention follows the laboratory specification:

- $0$ = moving tool;
- $1$ = background.

### Analytical Scope

The analysis covers baseline reproduction, temporal background modeling, intensity normalization, spatial and spectral filtering, morphology, deterministic thresholding, EM/GMM comparison, field-of-view restriction, and sequence-level validation using SAD, MSE, and PSNR.

## 1. Validate Data and Ground-Truth Paths

The experiment is defined by ten fluoroscopic frames and paired manual annotations. A valid evaluation requires exact frame-to-mask correspondence and identical spatial dimensions.

For each frame index $t$, the ground-truth tool support is constructed from the union of guidewire and microcatheter annotations. Any path mismatch or shape inconsistency invalidates all downstream metrics, even if the numerical pipeline itself executes successfully.

Input validation is therefore treated as part of the experimental protocol rather than as a software-only concern.

## 2. Reproduce the Original Baseline Pipeline

The original laboratory method uses the first processed frame as the background reference. After Gaussian preprocessing and intensity normalization, the signed residual is

$$
D_t = -(I_t-B_0).
$$

This baseline establishes the reference against which every later modification is evaluated. Reproducing it unchanged is essential for causal attribution: an improvement can only be credited to a specific processing change if the baseline is fixed and measurable.

## 3. Replace the First-Frame Background with a Temporal Median

A single reference frame may contain transient structures that contaminate the background model. The temporal median provides a more robust estimate:

$$
B_{\mathrm{med}}(x,y)
=
\operatorname{median}_{t}\, I_t(x,y).
$$

Sparse moving tools affect a pixel in only a subset of frames, so the median tends to preserve persistent anatomy while suppressing transient foreground structures.

The comparison with the first-frame baseline must be performed on the same evaluable frames. Frame 201, which is unusable as a test frame in the first-frame baseline, can be evaluated separately under the temporal-median model.

## 4. Stabilize the Pre-Subtraction Histogram Transformation

Independent frame-wise contrast normalization can create artificial temporal differences. To preserve comparability, the intensity mapping is estimated once from the background and then applied identically to every frame.

For fixed limits $L$ and $H$,

$$
I'(x,y)
=
\operatorname{clip}
\left(
\frac{I(x,y)-L}{H-L},
0,
1
\right).
$$

The purpose is not cosmetic enhancement. It is to enforce a common radiometric scale before subtraction so that residuals represent temporal change rather than normalization variability.

## 5. Optimize Spatial Gaussian Filtering

Gaussian smoothing reduces high-frequency acquisition noise before temporal subtraction:

$$
G_\sigma(x,y)
=
\frac{1}{2\pi\sigma^2}
\exp
\left(
-\frac{x^2+y^2}{2\sigma^2}
\right).
$$

The filtered frame is

$$
I_t^{(\sigma)}
=
G_\sigma * I_t.
$$

The main design trade-off is between noise suppression and preservation of thin interventional structures. For each candidate $\sigma$, the background must be rebuilt in the same filtered domain to avoid comparing signals processed under incompatible operators.

## 6. Add and Tune Spectral-Domain High-Pass Filtering

Residual anatomy and slowly varying background structure can persist after temporal subtraction. Frequency-domain filtering provides a complementary suppression mechanism.

Let

$$
F_t(u,v)=\mathcal{F}\{D_t(x,y)\}.
$$

A Gaussian high-pass transfer function can be written as

$$
H_{\mathrm{HP}}(u,v)
=
1-
\exp
\left(
-\frac{D(u,v)^2}{2D_0^2}
\right),
$$

where $D(u,v)$ is the distance from the centered spectrum origin.

The filtered residual is

$$
\widetilde{D}_t
=
\mathcal{F}^{-1}
\left\{
H_{\mathrm{HP}}F_t
\right\}.
$$

The cutoff $D_0$ controls the balance between removal of slowly varying background content and preservation of thin tool responses.

## 7. Optimize Morphological Refinement

Morphological operators regularize the spatial support of the residual response. Dilation expands selected structures, while opening removes small isolated components.

For binary set $A$ and structuring element $S$,

$$
A\oplus S
=
\{z \mid (\hat S)_z\cap A\neq\emptyset\},
$$

and

$$
A\circ S
=
(A\ominus S)\oplus S.
$$

Because guidewires and microcatheters are thin structures, excessive morphology can erase valid signal or artificially enlarge detections. Structuring-element size is therefore tuned under a fixed upstream configuration.

## 8. Optimize the Segmentation Threshold

After preprocessing, the residual feature image is converted to a binary tool decision using

$$
M_{\mathrm{tool}}(x,y)
=
\mathbf{1}
\left[
R(x,y)>T
\right].
$$

The threshold $T$ is not independent of the upstream pipeline. Any change in normalization, filtering, or morphology alters the feature distribution and can shift the optimal decision boundary.

A coarse-to-fine threshold sweep is therefore used to characterize sensitivity and retain a stable operating point rather than a visually chosen value.

## 9. Compare Threshold Segmentation with EM/GMM

A two-component Gaussian Mixture Model provides a probabilistic alternative to deterministic thresholding:

$$
p(x)
=
\sum_{k=1}^{2}
\pi_k
\mathcal{N}
\left(
x\mid\mu_k,\Sigma_k
\right).
$$

Expectation Maximization alternates between posterior assignment of samples to components and parameter re-estimation.

In this experiment, the component with the larger mean residual response is interpreted as tool-like. However, the tool class occupies only a small fraction of the image. Under strong class imbalance, the second Gaussian may model anatomical residuals or high-amplitude noise rather than the sparse tool distribution.

The GMM branch is therefore evaluated as an alternative hypothesis, not assumed to be superior because of greater statistical complexity.

## 10. Restrict Processing to a Valid Field-of-View Mask

Pixels outside the useful fluoroscopic field of view can create irrelevant detections. A support mask $Q$ is introduced to suppress those regions.

The principal guardrail is ground-truth coverage:

$$
\mathrm{Coverage}_t
=
\frac{|Q\cap G_t|}{|G_t|}.
$$

A candidate field-of-view mask is accepted only if

$$
\min_t \mathrm{Coverage}_t \approx 1.
$$

This prevents artificial metric improvement caused by removing true tool pixels from the evaluation domain.

## 11. Assemble the Final Retained Pipeline

Only processing stages supported by the controlled ablation study are retained.

The final sequence is

$$
\boxed{
I_t
\rightarrow
G_\sigma * I_t
\rightarrow
\text{fixed intensity mapping}
\rightarrow
(I_t-B_{\mathrm{med}})
\rightarrow
H_{\mathrm{HP}}
\rightarrow
\text{morphology}
\rightarrow
T
\rightarrow
Q
\rightarrow
M_t
}
$$

The rejected GMM experiment remains part of the evidence base but is excluded from the retained implementation branch.

## 12. Compute Sequence-Level Quantitative Evaluation

For ground truth $G_t$ and prediction $M_t$, the laboratory reports pixel-domain error measures.

The mean absolute difference used by the implementation is

$$
\mathrm{SAD}_t
=
\frac{1}{N}
\sum_{i=1}^{N}
|G_{t,i}-M_{t,i}|.
$$

The mean squared error is

$$
\mathrm{MSE}_t
=
\frac{1}{N}
\sum_{i=1}^{N}
(G_{t,i}-M_{t,i})^2.
$$

For peak value $L$,

$$
\mathrm{PSNR}_t
=
20\log_{10}
\left(
\frac{L}{\sqrt{\mathrm{MSE}_t}}
\right).
$$

Sequence-level mean and standard deviation summarize central performance and temporal variability. Per-frame curves remain necessary because a favorable mean can conceal isolated failures.

## 13. Run Numerical and Output-file Validation Checks

Execution alone does not establish experimental validity. The final validation stage verifies:

- completeness of frame and annotation sets;
- valid array shapes and finite values;
- binary mask convention;
- finite metric arrays with one value per evaluated frame;
- consistency of retained parameters;
- field-of-view coverage of annotated tool pixels;
- presence of the expected diagnostic figures.

The final success condition is therefore a conjunction of numerical validity, experimental consistency, and reproducible output generation.

## Technical Synthesis

The retained foreground-extraction model is supported by the following experimentally validated chain:

$$
\boxed{
\text{frames + annotations}
\rightarrow
B_{\mathrm{med}}
\rightarrow
\text{fixed radiometric mapping}
\rightarrow
\text{spatial filtering}
\rightarrow
\text{temporal residual}
\rightarrow
\text{spectral filtering}
\rightarrow
\text{morphology}
\rightarrow
\text{threshold + FOV}
\rightarrow
\text{sequence-level metrics}
}
$$

The distinguishing feature of the project is the controlled-ablation methodology: one design variable changes at a time, each candidate is evaluated under identical downstream conditions, and only evidence-supported changes are propagated into the retained branch.

## Scope and Limitations

### Included

- first-frame and temporal-median background models;
- fixed histogram mapping;
- Gaussian spatial filtering;
- Gaussian spectral high-pass filtering;
- morphology;
- deterministic threshold segmentation;
- EM/GMM comparison;
- field-of-view masking;
- SAD, MSE, and PSNR evaluation;
- qualitative mask and overlay inspection.

### Not included

- non-rigid registration;
- optical flow;
- learned segmentation networks;
- adaptive online background models;
- uncertainty calibration;
- clinical validation or deployment.

The method assumes that most anatomical content is sufficiently stable for temporal background modeling. Performance may degrade under substantial non-rigid motion, strong illumination drift, or tool occupancy patterns that violate the temporal-median assumption.